# Lottery Number Prediction Analysis: Statistical Foundations & Machine Learning

### Author / Context: AI/ML Engineering Study
**Objective:** Investigate whether machine learning models (Random Forest, Logistic Regression, Neural Networks) can predict future lottery outcomes based on historical draw data.

---

### Key Questions for AI/ML Students:
1. Are lottery draws independent and identically distributed (I.I.D.) random variables?
2. Can machine learning algorithms find hidden patterns or non-linear dependencies in truly random sequence data?
3. What is the difference between genuine predictive signal and model overfitting on random noise?
4. What mathematical theorems govern random draws (Law of Large Numbers, Information Theory / Entropy)?


In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.graphics.tsaplots import plot_acf

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, log_loss
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input

# Set random seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Libraries imported successfully.")


## 1. Simulating Historical Lottery Draw Data

In a standard Pick-1 lottery draw (or 1-digit number draw from $0$ to $9$), each draw outcome $X_t \in \{0, 1, 2, \dots, 9\}$. 
In a fair lottery mechanism (e.g., physical ball machines or cryptographically secure pseudorandom generators):
$$\mathbb{P}(X_t = k) = \frac{1}{10} = 0.1 \quad \forall k \in \{0, \dots, 9\}$$

Let's simulate $N = 5000$ consecutive daily lottery draws.


In [ ]:
# Simulate 5000 daily lottery draws (numbers 0 through 9)
N_DRAWS = 5000
N_CLASSES = 10

dates = pd.date_range(start='2010-01-01', periods=N_DRAWS, freq='D')
draws = np.random.randint(low=0, high=N_CLASSES, size=N_DRAWS)

df = pd.DataFrame({
    'Date': dates,
    'Winning_Number': draws
})

print(f"Dataset Shape: {df.shape}")
print("\nFirst 10 Lottery Draws:")
print(df.head(10))


## 2. Exploratory Data Analysis & Statistical Randomness Tests

Before training ML models, an AI/ML engineer must verify fundamental statistical properties of the data:
1. **Uniform Frequency Distribution:** Are all numbers drawn with equal probability? (Tested using Pearson's Chi-Square $\chi^2$ Goodness-of-Fit Test).
2. **Serial Independence:** Does draw $X_t$ depend on past draws $X_{t-1}, X_{t-2}, \dots$? (Tested using Autocorrelation Function - ACF).


In [ ]:
# 1. Frequency Analysis
counts = df['Winning_Number'].value_counts().sort_index()
expected_count = N_DRAWS / N_CLASSES

print("--- Frequency Count of Each Number ---")
for num, count in counts.items():
    print(f"Number {num}: {count} times ({count/N_DRAWS*100:.2f}%)")

# Chi-Square Goodness of Fit Test
chi2_stat, p_val = stats.chisquare(counts)
print(f"\nChi-Square Statistic: {chi2_stat:.4f}")
print(f"p-value: {p_val:.4f}")

if p_val > 0.05:
    print("Result: Fail to reject H0 -> Distribution is uniform (Fair Lottery).")
else:
    print("Result: Reject H0 -> Distribution is significantly non-uniform.")

# Plot Frequency
plt.figure(figsize=(10, 4))
bars = plt.bar(counts.index, counts.values, color='skyblue', edgecolor='navy')
plt.axhline(expected_count, color='red', linestyle='--', label=f'Expected Uniform Count ({int(expected_count)})')
plt.xlabel('Lottery Number')
plt.ylabel('Frequency')
plt.title('Distribution of Winning Lottery Numbers (5,000 Draws)')
plt.xticks(range(10))
plt.legend()
plt.show()


In [ ]:
# 2. Autocorrelation Analysis (Checking for sequence dependence / serial correlation)
plt.figure(figsize=(10, 4))
plot_acf(df['Winning_Number'], lags=30, alpha=0.05)
plt.title('Autocorrelation of Lottery Sequence (Lags 1-30)')
plt.xlabel('Lag (Days Prior)')
plt.ylabel('Autocorrelation coefficient')
plt.show()


## 3. Feature Engineering: Creating Historical Lag Features

To train machine learning models to predict tomorrow's lottery draw ($X_t$), we build features from previous consecutive draws:
$$\mathbf{X}_t = [X_{t-1}, X_{t-2}, X_{t-3}, \dots, X_{t-L}]$$

Where $L$ is the number of historical lookback lags (e.g., $L = 10$ past draws).


In [ ]:
# Create lag features (past L days)
LOOKBACK_LAGS = 10

df_ml = df.copy()
for lag in range(1, LOOKBACK_LAGS + 1):
    df_ml[f'lag_{lag}'] = df_ml['Winning_Number'].shift(lag)

# Drop missing rows due to lagging
df_ml.dropna(inplace=True)

feature_cols = [f'lag_{lag}' for lag in range(1, LOOKBACK_LAGS + 1)]
X = df_ml[feature_cols].values
y = df_ml['Winning_Number'].values

print(f"Feature matrix X shape: {X.shape}")
print(f"Target vector y shape: {y.shape}")

# Train-Test Split (Sequential / Time-Series split)
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set: {X_test.shape[0]} samples")


## 4. Machine Learning Experiments

We train three distinct model architectures:
1. **Baseline Random Guessing:** Theoretical accuracy $= 1/10 = 10.0\%$ (Random guessing according to prior probability).
2. **Logistic Regression (Linear / Softmax Classifier):** Checks for linear combinations of past lags.
3. **Random Forest Classifier (Ensemble Trees):** Checks for complex non-linear interaction rules between past draws.
4. **Deep Neural Network (Multi-Layer Perceptron):** Checks for deep non-linear feature representations.


In [ ]:
# 1. Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)
print(f"Random Forest Test Accuracy: {acc_rf * 100:.2f}%")

# 2. Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)
acc_lr = accuracy_score(y_test, y_pred_lr)
print(f"Logistic Regression Test Accuracy: {acc_lr * 100:.2f}%")


In [ ]:
# 3. Deep Neural Network
# One-hot encode targets for Keras categorical crossentropy
y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes=N_CLASSES)
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes=N_CLASSES)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

nn_model = Sequential([
    Input(shape=(LOOKBACK_LAGS,)),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(N_CLASSES, activation='softmax')
])

nn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history = nn_model.fit(
    X_train_scaled, y_train_cat,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    verbose=0
)

loss_nn, acc_nn = nn_model.evaluate(X_test_scaled, y_test_cat, verbose=0)
print(f"Neural Network Test Accuracy: {acc_nn * 100:.2f}%")


In [ ]:
# Model Comparison Table
baseline_acc = 1.0 / N_CLASSES

results_df = pd.DataFrame({
    'Model': ['Theoretical Random Baseline', 'Logistic Regression', 'Random Forest', 'Deep Neural Network'],
    'Test Accuracy (%)': [baseline_acc * 100, acc_lr * 100, acc_rf * 100, acc_nn * 100]
})

print("\n--- Performance Summary ---")
print(results_df.to_string(index=False))

# Plot Comparison
plt.figure(figsize=(8, 4))
bars = plt.bar(results_df['Model'], results_df['Test Accuracy (%)'], color=['gray', 'mediumseagreen', 'cornflowerblue', 'coral'])
plt.axhline(10.0, color='red', linestyle='--', label='Random Baseline (10.0%)')
plt.ylabel('Test Accuracy (%)')
plt.title('Model Test Accuracy vs. Theoretical Baseline')
plt.ylim(0, 20)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.5, f"{yval:.2f}%", ha='center', va='bottom')
plt.legend()
plt.tight_layout()
plt.show()


## 5. Theoretical Analysis & Key Takeaways for AI/ML Students

### Why Can't Machine Learning Predict Tomorrow's Lottery Number?

#### 1. Independent and Identically Distributed (I.I.D.) Random Variables
In a fair lottery, each draw $X_t$ is an **independent random variable**. 
Mathematical definition of independence:
$$\mathbb{P}(X_t = k \mid X_{t-1}, X_{t-2}, \dots, X_1) = \mathbb{P}(X_t = k) = \frac{1}{K}$$
The conditional probability distribution given all past history is identical to the unconditional prior distribution. There is **zero mutual information** between past draws and future draws:
$$I(X_t ; X_{t-1}, \dots, X_{t-L}) = 0$$

#### 2. The Gambler's Fallacy
A common misconception is that if a number hasn't appeared in a long time, it is "due" to appear ("hot" or "cold" numbers). Because draws are memoryless:
$$\mathbb{P}(\text{Number } 7 \text{ tomorrow} \mid \text{7 hasn't appeared in 50 days}) = \frac{1}{10}$$
Past states provide **no causal or statistical leverage**.

#### 3. Overfitting on Noise vs. Learning Real Signal
If a complex ML model (e.g. deep tree or deep neural net) achieves >10% accuracy on training data, it is merely fitting random noise (**spurious correlations**). When evaluated on unseen test data, accuracy reverts back to ~10%.

#### 4. Practical Takeaway for ML Engineers
Machine Learning models require **predictive signal (patterns, dependencies, structures)** in the features. When applied to purely stochastic system outputs, ML models cannot outperform random baseline guessing.
